# task_20 Solution

In [ ]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_20/data/")


In [ ]:

fields = pd.read_csv(base / "fields.csv")
plantings = pd.read_csv(base / "plantings.csv")
harvests = pd.read_csv(base / "harvests.csv")
pests = pd.read_csv(base / "pests.csv")

Q2: Among fields whose yield per hectare is below their crop's average, compute rainfall efficiency (yield_tons / cumulative_rain_mm from plant_date to end of weather data). Which field has highest rainfall efficiency?

In [ ]:
weather = pd.read_csv(base / "weather.csv")
weather["date"] = pd.to_datetime(weather["date"])
plantings["plant_date"] = pd.to_datetime(plantings["plant_date"])
harvests["harvest_date"] = pd.to_datetime(harvests["harvest_date"])

yield_df = fields.merge(harvests, on="field_id").merge(plantings[["field_id", "plant_date"]], on="field_id")
yield_df["yield_per_ha"] = yield_df["yield_tons"] / yield_df["area_ha"]

# Identify underperforming fields (below crop average yield/ha)
crop_avg = yield_df.groupby("crop")["yield_per_ha"].mean().rename("crop_avg")
yield_df = yield_df.join(crop_avg, on="crop")
underperformers = yield_df[yield_df["yield_per_ha"] < yield_df["crop_avg"]].copy()

# Compute cumulative rainfall from plant_date to last weather date
last_weather_date = weather["date"].max()
def cum_rain(row):
    mask = (weather["date"] >= row["plant_date"]) & (weather["date"] <= last_weather_date)
    return weather.loc[mask, "rain_mm"].sum()

underperformers["cum_rain"] = underperformers.apply(cum_rain, axis=1)
underperformers["rain_eff"] = underperformers["yield_tons"] / underperformers["cum_rain"]

q2 = underperformers.sort_values(["rain_eff", "field_id"], ascending=[False, True]).iloc[0]["field_id"]
print(q2)

Q5: What percentage of fields were harvested after targeted date and also had yield per hectare below the median yield per hectare of their crop, rounded to 1 decimal place?

In [ ]:
late_df = fields.merge(harvests, on="field_id").merge(plantings[["field_id", "target_harvest_date"]], on="field_id")
late_df["yield_per_ha"] = late_df["yield_tons"] / late_df["area_ha"]
crop_median = late_df.groupby("crop")["yield_per_ha"].median().rename("crop_median")
late_df = late_df.join(crop_median, on="crop")
condition = (late_df["harvest_date"] > late_df["target_harvest_date"]) & (late_df["yield_per_ha"] < late_df["crop_median"])
q5 = f"{(100 * condition.mean()):.1f}%"
print(q5)
